# 02: Preprocess + cache

Resample every recording to 16 kHz mono, segment respiratory cycles using the official annotations, cyclic-pad each cycle to 8 s (128000 samples), and cache the result to a single `.pt` file.

Cache layout: `{x_train, y_train, device_train, stem_train, x_test, y_test, device_test, stem_test, sample_rate, target_samples, device_to_id}`.

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/safina57/project-deep-learning.git"

def _find_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for p in (cwd, *cwd.parents):
        if (p / "pyproject.toml").exists():
            return p
    return None

ROOT = _find_root()
if ROOT is None:
    if "google.colab" in sys.modules:
        target = Path("/content/repo")
        if not target.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
        os.chdir(target)
        ROOT = _find_root()
    if ROOT is None:
        raise RuntimeError(
            "Project root not found. Locally: launch from inside the cloned repo. "
            "On Colab: ensure git is available and the repo URL is reachable."
        )
sys.path.insert(0, str(ROOT))
print(f"project root: {ROOT}")

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "librosa", "soundfile", "python-dotenv", "kaggle"],
        check=True,
    )

project root: /content/repo


## Data bootstrap (Colab only)

Mounts Drive (where the cache will be written), pulls Kaggle credentials from Colab Secrets, downloads ICBHI if missing, fetches `train_test.txt`.

In [2]:
import urllib.request

SPLIT_URL = (
    "https://raw.githubusercontent.com/joetho786/"
    "Respiratory-Sound-Classification-in-Wearable-Devices-Enabled-by-Patient-Specific-Model-Tuning/"
    "main/ICBHI_challenge_train_test.txt"
)

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
    os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))

    LOCAL_DATA = ROOT / "data" / "icbhi"
    ANNOT_DIR = LOCAL_DATA / "ICBHI_final_database"
    if not (ANNOT_DIR.exists() and any(ANNOT_DIR.iterdir())):
        subprocess.run([sys.executable, str(ROOT / "scripts/download_icbhi.py")], check=True)
    if not (ANNOT_DIR / "train_test.txt").exists():
        urllib.request.urlretrieve(SPLIT_URL, ANNOT_DIR / "train_test.txt")
        print("fetched train_test.txt")
    os.environ["ICBHI_ROOT"] = str(LOCAL_DATA)

Mounted at /content/drive
fetched train_test.txt


## Output path

In [3]:
from src.data.paths import detect_runtime

runtime = detect_runtime()
if runtime == "colab":
    OUT_PATH = Path("/content/drive/MyDrive/icbhi_cache/icbhi_16k_8s.pt")
elif runtime == "kaggle":
    OUT_PATH = Path("/kaggle/working/icbhi_16k_8s.pt")
else:
    OUT_PATH = ROOT / "data" / "cache" / "icbhi_16k_8s.pt"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"runtime:  {runtime}")
print(f"out path: {OUT_PATH}")

runtime:  colab
out path: /content/drive/MyDrive/icbhi_cache/icbhi_16k_8s.pt


## Run preprocessing

In [4]:
from src.data.preprocessing import build_default, summarize

cache = build_default(out_path=OUT_PATH)
print(summarize(cache))
print(f"saved: {OUT_PATH}")

preprocess:   0%|          | 0/920 [00:00<?, ?it/s]

train: x=(4142, 128000) torch.float32  Normal=2063  Crackle=1215  Wheeze=501  Both=363
test: x=(2756, 128000) torch.float32  Normal=1579  Crackle=649  Wheeze=385  Both=143
saved: /content/drive/MyDrive/icbhi_cache/icbhi_16k_8s.pt


## Reload sanity

In [5]:
import torch
from src.data.annotations import CLASS_NAMES

reloaded = torch.load(OUT_PATH, weights_only=False)
x_tr, y_tr = reloaded["x_train"], reloaded["y_train"]
x_te, y_te = reloaded["x_test"], reloaded["y_test"]
print(f"train: {tuple(x_tr.shape)}  test: {tuple(x_te.shape)}")
assert x_tr.shape[1] == 128000 and x_tr.dtype == torch.float32
assert x_te.shape[1] == 128000 and x_te.dtype == torch.float32

total = x_tr.shape[0] + x_te.shape[0]
print(f"total cycles: {total}  (audit reported 6898)")
for k, name in enumerate(CLASS_NAMES):
    n = int((y_tr == k).sum()) + int((y_te == k).sum())
    print(f"  {name:8s} {n}")

train: (4142, 128000)  test: (2756, 128000)
total cycles: 6898  (audit reported 6898)
  Normal   3642
  Crackle  1864
  Wheeze   886
  Both     506
